In [0]:
import pandas as pd
import numpy as np

RAW_PATH = "/Volumes/education_analytics/default/education_analytics_platform/enrollment_data_raw.csv"

df = pd.read_csv(RAW_PATH)
display(df.head())

In [0]:
print("=" * 55)
print("DATASET LOADED SUCCESSFULLY")
print("=" * 55)
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")
print(f"\nColumn names:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:>2}. {col}")

In [0]:
#Schema and Data Types
print("DATA TYPES")
print("-" * 45)
print(df.dtypes.to_string())

print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
print(f"\nFirst 5 rows:")
display(df.head())

In [0]:
# Missing Values Analysis

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    "Column": missing.index,
    "Missing Count": missing.values,
    "Missing %": missing_pct.values
}).query("`Missing Count` > 0").sort_values("Missing Count", ascending=False)

print("MISSING VALUES SUMMARY")
print("-" * 45)
if missing_df.empty:
    print("No missing values found")
else:
    print(missing_df.to_string(index=False))

total_missing = missing.sum()
print(f"\nTotal missing cells : {total_missing:,}")
print(f"Total cells         : {df.shape[0] * df.shape[1]:,}")
print(f"Overall missing %   : {total_missing / (df.shape[0] * df.shape[1]) * 100:.2f}%")


In [0]:
# 4 — Duplicate Detection

# Exact duplicates — all columns identical
exact_dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {exact_dupes:,}")

# Business key duplicates — same school + year + grade + subject
key_cols = ["school_name", "academic_year", "grade", "subject"]
biz_dupes = df.duplicated(subset=key_cols).sum()
print(f"Business key duplicates (school+year+grade+subject): {biz_dupes:,}")

if exact_dupes > 0:
    print(f"\nSample exact duplicates:")
    display(df[df.duplicated(keep=False)].head(6))

In [0]:
# 5 — Outlier Detection

# COMMAND ----------

numeric_cols = ["total_enrollment", "avg_performance_score", "dropout_rate",
                "male_count", "female_count", "pass_rate", "budget_per_student_usd"]

print("OUTLIER SUMMARY")
print("-" * 70)

for col in numeric_cols:
    if col in df.columns:
        series = pd.to_numeric(df[col], errors="coerce").dropna()
        q1, q3 = series.quantile(0.25), series.quantile(0.75)
        iqr = q3 - q1
        outliers = series[(series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)]
        print(f"{col:<30} min={series.min():.1f}  max={series.max():.1f}  "
              f"mean={series.mean():.1f}  outliers={len(outliers)}")

# Specific business rule violations
print("\nBUSINESS RULE VIOLATIONS")
print("-" * 70)
df_num = df.copy()
df_num["avg_performance_score"] = pd.to_numeric(df_num["avg_performance_score"], errors="coerce")
df_num["dropout_rate"]          = pd.to_numeric(df_num["dropout_rate"], errors="coerce")
df_num["total_enrollment"]      = pd.to_numeric(df_num["total_enrollment"], errors="coerce")

invalid_score    = df_num[(df_num["avg_performance_score"] < 0) | (df_num["avg_performance_score"] > 100)]
invalid_dropout  = df_num[(df_num["dropout_rate"] < 0) | (df_num["dropout_rate"] > 1)]
invalid_enroll   = df_num[df_num["total_enrollment"] <= 0]

print(f"Performance score out of 0-100 range : {len(invalid_score):,} rows")
print(f"Dropout rate out of 0-1 range        : {len(invalid_dropout):,} rows")
print(f"Enrollment <= 0                      : {len(invalid_enroll):,} rows")


In [0]:
# 6 — Inconsistent Naming and Text Issues

# COMMAND ----------

print("SCHOOL NAME INCONSISTENCIES")
print("-" * 55)
school_names = df["school_name"].dropna().unique()
print(f"Unique school name values: {len(school_names)}")
print("\nSample (first 30):")
for name in sorted(school_names)[:30]:
    print(f"  '{name}'")

print("\n\nREGION VALUES")
print("-" * 55)
regions = df["region"].dropna().unique()
print(f"Unique regions: {len(regions)}")
for r in sorted(regions):
    print(f"  '{r}'")

print("\n\nGRADE VALUES")
print("-" * 55)
grades = df["grade"].dropna().unique()
print(f"Unique grade values: {len(grades)}")
for g in sorted(grades):
    print(f"  '{g}'")

print("\n\nSUBJECT VALUES")
print("-" * 55)
subjects = df["subject"].dropna().unique()
for s in sorted(subjects):
    print(f"  '{s}'")

In [0]:
# 7 — Year and Type Inconsistencies

# COMMAND ----------

print("ACADEMIC YEAR VALUES")
print("-" * 45)
years = df["academic_year"].dropna().unique()
print(f"Unique academic_year values: {len(years)}")
for y in sorted(years):
    print(f"  '{y}'")

print("\n\nDATA QUALITY FLAG DISTRIBUTION")
print("-" * 45)
if "data_quality_flag" in df.columns:
    flag_counts = df["data_quality_flag"].value_counts()
    print(flag_counts.to_string())
    print(f"\nTotal flagged rows: {df['data_quality_flag'].notna().sum():,}")
    print(f"Clean rows        : {df['data_quality_flag'].isna().sum():,}")


In [0]:
# 8 — Enrollment Consistency Check

# COMMAND ----------

df_check = df.copy()
df_check["male_count"]       = pd.to_numeric(df_check["male_count"], errors="coerce")
df_check["female_count"]     = pd.to_numeric(df_check["female_count"], errors="coerce")
df_check["total_enrollment"] = pd.to_numeric(df_check["total_enrollment"], errors="coerce")

df_check["expected_total"] = df_check["male_count"] + df_check["female_count"]
df_check["total_mismatch"] = abs(df_check["total_enrollment"] - df_check["expected_total"]) > 1

mismatches = df_check[df_check["total_mismatch"] == True]
print(f"ENROLLMENT MATH INCONSISTENCIES")
print("-" * 55)
print(f"Rows where total ≠ male + female: {len(mismatches):,}")

if len(mismatches) > 0:
    print("\nSample mismatches:")
    display(mismatches[["school_name", "male_count", "female_count",
                         "total_enrollment", "expected_total"]].head(10))


In [0]:
# 9 — Distribution Analysis by Year and Region

# COMMAND ----------

df_clean_num = df.copy()
df_clean_num["total_enrollment"]      = pd.to_numeric(df_clean_num["total_enrollment"], errors="coerce")
df_clean_num["avg_performance_score"] = pd.to_numeric(df_clean_num["avg_performance_score"], errors="coerce")
df_clean_num["dropout_rate"]          = pd.to_numeric(df_clean_num["dropout_rate"], errors="coerce")

print("ENROLLMENT BY ACADEMIC YEAR (raw data)")
print("-" * 55)
yr = df_clean_num.groupby("academic_year")["total_enrollment"].agg(["sum","mean","count"])
yr.columns = ["Total Enrollment", "Avg Enrollment", "Records"]
print(yr.to_string())

print("\n\nAVG PERFORMANCE BY REGION (raw data)")
print("-" * 55)
reg = df_clean_num.groupby("region")["avg_performance_score"].agg(["mean","count"])
reg.columns = ["Avg Score", "Records"]
print(reg.sort_values("Avg Score", ascending=False).to_string())

print("\n\nDROPOUT RATE STATISTICS")
print("-" * 55)
print(df_clean_num["dropout_rate"].describe().round(4).to_string())
